# 훈련 노트북

**최초 1회**: `download_to_drive.ipynb` 먼저 실행 → Drive에 `processed_data.tar.gz` 생성

**이후 매 세션**: 이 노트북만 사용

```
Step 0:  환경 설정 (항상 먼저)
Step 1:  Drive tar.gz → 로컬 복원
Step 2a: Alpha 탐색 (α ∈ {0.3, 0.5, 1.0} × A-soft × seed 0)
Step 2b: Alpha 비교 → 최적 α 결정
Step 2c: 본 훈련 (5 ablation × 3 seeds = 15회)
Step 3:  결과 Drive 저장
```

> ⚠️ **커널 재시작 후에는 Step 0부터 다시 실행하세요.**
>
> ⚠️ 레포가 **Private**이면: Runtime → Manage secrets → `GITHUB_TOKEN` 추가

---
## Step 0. 환경 설정 (항상 가장 먼저 실행)

In [ ]:
import os, sys, subprocess

# ── 레포 설정 ──
GITHUB_USER = 'sungmin-park-dev'
REPO_NAME   = 'tactical-speech-enhancement'
REPO_DIR    = f'/content/{REPO_NAME}'
DRIVE_ROOT  = '/content/drive/MyDrive/tactical-speech-enhancement'
COLAB_CFG   = '/content/data_config_colab.yaml'

# ── 1) 레포 클론 ──
if not os.path.isdir(REPO_DIR):
    token = os.getenv('GITHUB_TOKEN', '')
    if token:
        clone_url = f'https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
    else:
        clone_url = f'https://github.com/{GITHUB_USER}/{REPO_NAME}.git'
    print(f'클론 중: github.com/{GITHUB_USER}/{REPO_NAME}')
    result = subprocess.run(['git', 'clone', clone_url, REPO_DIR],
                            capture_output=True, text=True)
    if result.returncode != 0:
        print(f'❌ git clone 실패 (exit {result.returncode})')
        print(f'   stderr: {result.stderr.strip()}')
        if not token:
            print('\n💡 레포가 Private이면 토큰이 필요합니다:')
            print('   Runtime → Manage secrets → GITHUB_TOKEN 추가')
        raise RuntimeError('git clone 실패')
    print('✅ 레포 클론 완료')
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--quiet'], check=True)
    print('✅ 레포 최신화 완료')

# ── 2) sys.path 등록 ──
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# ── 3) 패키지 설치 ──
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'soundfile', 'librosa', 'pesq', 'pystoi', 'pyyaml', 'tqdm', 'seaborn'],
    check=True
)

# ── 4) GPU / 디스크 확인 ──
import torch
print(f'\nGPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음"}')
if torch.cuda.is_available():
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
!df -h /content | tail -1
print('\n✅ Step 0 완료')

---
## Step 1. 데이터 복원

Drive의 `processed_data.tar.gz` → `/content/data/processed/` 로컬 압축 해제

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ARCHIVE = f'{DRIVE_ROOT}/data/processed_data.tar.gz'

if os.path.exists('/content/data/processed'):
    print('processed 데이터 이미 로컬에 있음 — 스킵')
elif os.path.exists(DRIVE_ARCHIVE):
    print('Drive에서 복원 중...')
    !cp {DRIVE_ARCHIVE} /tmp/processed_data.tar.gz
    !mkdir -p /content/data && tar -xzf /tmp/processed_data.tar.gz -C /content/data/
    !rm /tmp/processed_data.tar.gz
    print('복원 완료')
else:
    print('⚠ processed_data.tar.gz 없음 → download_to_drive.ipynb 먼저 실행하세요')

!du -sh /content/data/processed/* 2>/dev/null || echo '데이터 없음'

In [ ]:
# Colab config 생성
import yaml

with open(f'{REPO_DIR}/configs/data_config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['paths']['output_root'] = '/content/data/processed'
with open(COLAB_CFG, 'w') as f:
    yaml.dump(cfg, f, allow_unicode=True)
print('config 준비 완료:', COLAB_CFG)

---
## Step 2a. Alpha 탐색 (최초 1회)

A-soft × seed 0 으로 α ∈ {0.3, 0.5, 1.0} 비교 → 최적 α 결정

In [ ]:
# Alpha 탐색: A-soft / seed=0 / α ∈ {0.3, 0.5, 1.0}
import torch

ENV   = 'military'
BATCH = 16 if 'A100' in (torch.cuda.get_device_name(0) if torch.cuda.is_available() else '') else 8

for alpha in [0.3, 0.5, 1.0]:
    save_dir = f'/content/results/dpcrn/alpha_search/alpha{alpha}'
    print(f'\n{"="*60}')
    print(f'Alpha 탐색: α={alpha}')
    print(f'{"="*60}')
    !python {REPO_DIR}/train.py \
        --config     {REPO_DIR}/configs/train_config.yaml \
        --data_config {COLAB_CFG} \
        --ablation   A-soft \
        --env        {ENV} \
        --alpha      {alpha} \
        --seed       0 \
        --epochs     50 \
        --batch_size {BATCH} \
        --save_dir   {save_dir}

print('\n✅ Alpha 탐색 완료 — Step 3에서 결과 저장 후 비교')

---
## Step 2b. Alpha 탐색 결과 비교

best.pt 체크포인트의 val_loss / val_SI-SNR 비교 → 최적 α 선택

In [ ]:
import torch

print(f'{"Alpha":>8} | {"val_loss":>10} | {"val_SI-SNR":>12}')
print('-' * 38)

best_alpha, best_loss = None, float('inf')
for alpha in [0.3, 0.5, 1.0]:
    ckpt_path = f'/content/results/dpcrn/alpha_search/alpha{alpha}/best.pt'
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
        vl = ckpt['val_loss']
        vs = ckpt.get('val_sisnr', float('nan'))
        print(f'{alpha:>8.1f} | {vl:>10.4f} | {vs:>10.2f} dB')
        if vl < best_loss:
            best_loss, best_alpha = vl, alpha
    else:
        print(f'{alpha:>8.1f} | {"N/A":>10} | {"N/A":>12}')

print(f'\n>>> 최적 α = {best_alpha} (val_loss = {best_loss:.4f})')
print('아래 Step 2c에서 BEST_ALPHA 변수를 이 값으로 설정하세요.')

---
## Step 2c. DPCRN 본 훈련 (5 ablation × 3 seeds)

Alpha 탐색에서 결정한 최적 α로 15회 훈련 실행

In [ ]:
# ── 설정: Alpha 탐색 결과를 여기에 반영 ──
BEST_ALPHA = 0.5  # Step 2b 결과로 변경

ABLATIONS = ['A-soft', 'A-hard', 'A-param', 'B', 'C']
SEEDS     = [0, 1, 2]
ENV       = 'military'

import torch
BATCH = 16 if 'A100' in (torch.cuda.get_device_name(0) if torch.cuda.is_available() else '') else 8

total = len(ABLATIONS) * len(SEEDS)
done  = 0

for abl in ABLATIONS:
    for seed in SEEDS:
        done += 1
        save_dir = f'/content/results/dpcrn/{abl}/seed{seed}'

        # 이미 완료된 실험 스킵
        if os.path.exists(f'{save_dir}/best.pt'):
            print(f'[{done}/{total}] {abl}/seed{seed} — 이미 완료, 스킵')
            continue

        print(f'\n{"="*60}')
        print(f'[{done}/{total}] {abl} / seed={seed} / α={BEST_ALPHA}')
        print(f'{"="*60}')
        !python {REPO_DIR}/train.py \
            --config     {REPO_DIR}/configs/train_config.yaml \
            --data_config {COLAB_CFG} \
            --ablation   {abl} \
            --env        {ENV} \
            --alpha      {BEST_ALPHA} \
            --seed       {seed} \
            --epochs     50 \
            --batch_size {BATCH} \
            --save_dir   {save_dir}

print(f'\n✅ 본 훈련 완료 ({total}회) — Step 3에서 결과 저장')

---
## Step 3. 결과 Drive 저장

> Google 권장: 결과도 단일 tar.gz로 압축 후 Drive 복사

In [ ]:
import time

DRIVE_RESULTS = f'{DRIVE_ROOT}/results'
!mkdir -p {DRIVE_RESULTS}

if os.path.exists('/content/results'):
    TS = int(time.time())
    LOCAL_ARCHIVE = f'/tmp/results_{TS}.tar.gz'

    print('① 결과 압축 중...')
    !tar -czf {LOCAL_ARCHIVE} -C /content results
    !du -sh {LOCAL_ARCHIVE}

    print('② Drive로 단일 파일 복사 중...')
    !cp {LOCAL_ARCHIVE} {DRIVE_RESULTS}/results_{TS}.tar.gz
    !rm {LOCAL_ARCHIVE}

    print(f'\n✅ 저장 완료: {DRIVE_RESULTS}/results_{TS}.tar.gz')
else:
    print('저장할 결과 없음 (/content/results 없음)')

---
## [선택] 파이프라인 빠른 검증

데이터/모델 없이 파이프라인 동작 확인 (더미 신호 사용)

In [ ]:
import numpy as np
from data.bc_simulator import BCSimulator
from data.saturation import SaturationSimulator
from data.noise_mixer import NoiseMixer
from data.pipeline import SampleGenerator

FS  = 16000
rng = np.random.default_rng(0)
t   = np.linspace(0, 4, FS * 4, endpoint=False)
speech = (0.3 * np.sin(2 * np.pi * 200 * t)).astype('float32')

bc_sim  = BCSimulator(sample_rate=FS, rng=rng)
sat_sim = SaturationSimulator(sample_rate=FS, rng=rng)
n_mixer = NoiseMixer(env='military', sample_rate=FS, rng=rng)
gen     = SampleGenerator('military', bc_sim, sat_sim, n_mixer, rng=rng)

sample = gen.generate(speech)
print('bc_signal :', sample['bc_signal'].shape)
print('ac_noisy  :', sample['ac_noisy'].shape)
print('masks     :', {k: v.shape for k, v in sample['masks'].items()})
print('meta      :', sample['meta'])
print('\n✅ 파이프라인 검증 통과')